# PRODUCTION - local or GCP

In [6]:
from __future__ import annotations

import json
import logging
import re
from pathlib import Path
from typing import Optional

import dateparser
import pdfplumber
from requests.adapters import HTTPAdapter

try:
    from google.cloud import storage
except Exception:
    storage = None


# ---------- Configuration ----------
MAX_PAGES = 4
MAX_TRIES = 3
OUTPUT_DIR = Path.cwd()
OUTPUT_CSV = OUTPUT_DIR / "handin_month_summary.csv"
OUTPUT_JSON = OUTPUT_DIR / "handin_month_summary.json"
GCP_DEFAULT_BUCKET = "thesis_archive_bucket"
GCP_DEFAULT_PREFIX = "dtu_findit/master_thesis/"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("seasonality_extractor")

# Danish -> English normalization so output is always "Month YYYY" in English
MONTH_TRANSLATIONS = {
    "januar": "january",
    "februar": "february",
    "marts": "march",
    "april": "april",
    "maj": "may",
    "juni": "june",
    "juli": "july",
    "august": "august",
    "september": "september",
    "oktober": "october",
    "november": "november",
    "december": "december",
}

# Ordered by confidence: range dates first (must take latter date), then explicit month-year patterns.
PATTERNS = [
    # 29/10/2018-29/04/2019 (extract latter date)
    re.compile(
        r"\b"
        r"(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})"
        r"\s*(?:-|–|—|to)\s*"
        r"(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # June 2, 2019 / JUNE 2, 2019
    re.compile(
        r"\b"
        r"((?:[A-Za-zÀ-ÖØ-öø-ÿ]+)\s+\d{1,2},?\s+\d{4})"
        r"\b",
        re.IGNORECASE,
    ),
    # 2 June 2019
    re.compile(
        r"\b"
        r"(\d{1,2}\s+[A-Za-zÀ-ÖØ-öø-ÿ]+\s+\d{4})"
        r"\b",
        re.IGNORECASE,
    ),
    # Kongens Lyngby, June 2025 / Kongens Lyngby June 2025 (location is ignored by parser)
    re.compile(
        r"\b"
        r"(?:[A-Za-zÀ-ÖØ-öø-ÿ]+(?:\s+[A-Za-zÀ-ÖØ-öø-ÿ]+){0,3},\s*)?"
        r"([A-Za-zÀ-ÖØ-öø-ÿ]+\s+\d{4})"
        r"\b",
        re.IGNORECASE,
    ),
    # Numeric single date fallback
    re.compile(r"\b(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})\b"),
]


def normalize_text(text: str) -> str:
    """Lower-risk normalization to improve pattern matching and Danish month parsing."""
    cleaned = text.replace("\u00a0", " ")
    cleaned = re.sub(r"\s+", " ", cleaned)

    # Replace Danish month names with English names for consistent parsing/output.
    for dk, en in MONTH_TRANSLATIONS.items():
        cleaned = re.sub(rf"\b{dk}\b", en, cleaned, flags=re.IGNORECASE)

    return cleaned.strip()


def parse_to_month_year(date_text: str) -> str | None:
    """Parse a candidate date string to standardized 'Month YYYY'."""
    dt = dateparser.parse(
        date_text,
        languages=["en", "da"],
        settings={
            "DATE_ORDER": "DMY",
            "PREFER_DAY_OF_MONTH": "first",
            "PREFER_DATES_FROM": "past",
            "NORMALIZE": True,
        },
    )
    if not dt:
        return None
    return dt.strftime("%B %Y")


def extract_handin_month(text: str) -> str | None:
    normalized = normalize_text(text)

    for pattern in PATTERNS:
        for match in pattern.finditer(normalized):
            groups = [g for g in match.groups() if g]

            # Date range: always use latter date.
            if len(groups) == 2:
                candidate = groups[1]
                parsed = parse_to_month_year(candidate)
                if parsed:
                    return parsed
            else:
                candidate = groups[0] if groups else match.group(0)
                parsed = parse_to_month_year(candidate)
                if parsed:
                    return parsed

    return None


def extract_handin_month_from_pdf(
    pdf_path: Path, chunk_size: int = MAX_PAGES, max_tries: int = MAX_TRIES
) -> str | None:
    """Scan PDF in chunk_size-page windows and stop after max_tries windows."""
    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)

        for attempt_idx, start in enumerate(range(0, total_pages, chunk_size), start=1):
            if attempt_idx > max_tries:
                break

            chunk = pdf.pages[start : start + chunk_size]
            text = "\n".join((page.extract_text() or "") for page in chunk)
            parsed = extract_handin_month(text)
            if parsed:
                return parsed

    return None


def run_extraction(pdf_files: list[Path]) -> list[dict[str, str | None]]:
    results: list[dict[str, str | None]] = []

    for pdf_file in pdf_files:
        extracted_month = extract_handin_month_from_pdf(
            pdf_file, chunk_size=MAX_PAGES, max_tries=MAX_TRIES
        )
        results.append({"filename": pdf_file.name, "extracted_month": extracted_month})

    return results


class GCPBatchProcessor:
    """Process thesis PDFs in batches from GCS for seasonality extraction."""

    def __init__(
        self,
        bucket_name: str,
        gcs_prefix: str = "dtu_findit/master_thesis",
        local_pdf_folder: str = "thesis_pdfs",
        batch_size: int = 10,
    ):
        if storage is None:
            raise ImportError(
                "google-cloud-storage is not installed. Install it to use gcp mode."
            )

        self.bucket_name = bucket_name
        self.gcs_prefix = gcs_prefix
        self.local_pdf_folder = Path(local_pdf_folder)
        self.batch_size = batch_size

        self.local_pdf_folder.mkdir(exist_ok=True)

        logger.info("Initialising GCP Storage client...")
        self.client = storage.Client()
        self.bucket = self.client.bucket(bucket_name)
        self._configure_http_pool(max_pool_size=max(10, self.batch_size * 2))
        logger.info("Connected to bucket: gs://%s", bucket_name)

    def _configure_http_pool(self, max_pool_size: int) -> None:
        """Configure the underlying HTTP adapter pool size for GCS requests."""
        pool_size = max(10, int(max_pool_size))
        try:
            adapter = HTTPAdapter(
                pool_connections=pool_size,
                pool_maxsize=pool_size,
                max_retries=0,
            )
            self.client._http.mount("https://", adapter)
            self.client._http.mount("http://", adapter)
            logger.debug("Configured HTTP connection pool size: %d", pool_size)
        except Exception as e:
            logger.debug("Could not tune HTTP pool size: %s", e)

    def list_thesis_pdfs(self) -> list[str]:
        logger.info(f"Listing PDFs from gs://{self.bucket_name}/{self.gcs_prefix}/")
        blobs = self.client.list_blobs(self.bucket_name, prefix=self.gcs_prefix)
        pdf_paths = [blob.name for blob in blobs if blob.name.lower().endswith(".pdf")]
        logger.info(f"Found {len(pdf_paths)} PDF files")
        return pdf_paths

    def download_batch(self, pdf_paths: list[str], start_idx: int) -> list[Path]:
        end_idx = min(start_idx + self.batch_size, len(pdf_paths))
        batch = pdf_paths[start_idx:end_idx]

        logger.info(
            f"Downloading batch {start_idx // self.batch_size + 1}: "
            f"PDFs {start_idx + 1}-{end_idx} of {len(pdf_paths)}"
        )

        local_paths: list[Path] = []
        for gcs_path in batch:
            filename = Path(gcs_path).name
            local_path = self.local_pdf_folder / filename

            try:
                blob = self.bucket.blob(gcs_path)
                blob.download_to_filename(str(local_path))
                local_paths.append(local_path)
                logger.info(f"  Downloaded: {filename}")
            except Exception as e:
                logger.error(f"  Failed to download {filename}: {e}")

        return local_paths

    def extract_from_batch(self, pdf_paths: list[Path]) -> list[dict[str, str | None]]:
        results: list[dict[str, str | None]] = []
        logger.info(f"Extracting hand-in month from {len(pdf_paths)} PDFs...")

        for pdf_path in pdf_paths:
            try:
                extracted_month = extract_handin_month_from_pdf(
                    pdf_path, chunk_size=MAX_PAGES, max_tries=MAX_TRIES
                )
                results.append(
                    {"filename": pdf_path.name, "extracted_month": extracted_month}
                )

                if extracted_month:
                    logger.info(f"  Found date in: {pdf_path.name} -> {extracted_month}")
                else:
                    logger.info(f"  No date found in: {pdf_path.name}")

            except Exception as e:
                logger.error(f"  Error processing {pdf_path.name}: {e}")
                results.append({"filename": pdf_path.name, "extracted_month": None})

        return results

    def cleanup_batch(self, pdf_paths: list[Path]) -> None:
        for pdf_path in pdf_paths:
            try:
                pdf_path.unlink()
            except Exception as e:
                logger.warning(f"Failed to delete {pdf_path.name}: {e}")

    def process_all_batches(
        self, keep_pdfs: bool = False, max_batches: Optional[int] = None
    ) -> list[dict[str, str | None]]:
        all_pdf_paths = self.list_thesis_pdfs()
        if not all_pdf_paths:
            logger.warning("No PDFs found in bucket!")
            return []

        all_results: list[dict[str, str | None]] = []

        num_batches = (len(all_pdf_paths) + self.batch_size - 1) // self.batch_size
        if max_batches:
            num_batches = min(num_batches, max_batches)

        for batch_num in range(num_batches):
            start_idx = batch_num * self.batch_size
            logger.info(f"{'=' * 60}")
            logger.info(f"BATCH {batch_num + 1}/{num_batches}")
            logger.info(f"{'=' * 60}")

            local_paths = self.download_batch(all_pdf_paths, start_idx)
            batch_results = self.extract_from_batch(local_paths)
            all_results.extend(batch_results)

            if not keep_pdfs:
                self.cleanup_batch(local_paths)

        return all_results


def save_outputs(results: list[dict[str, str | None]]) -> None:
    # Save CSV manually to avoid extra dependency; JSON for easy downstream processing.
    header = "filename,extracted_month\n"
    rows = [
        f"{row['filename']},{'' if row['extracted_month'] is None else row['extracted_month']}"
        for row in results
    ]
    OUTPUT_CSV.write_text(header + "\n".join(rows) + "\n", encoding="utf-8")
    # OUTPUT_JSON.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")


# ==== SELECT MODE ====
print("Choose extraction mode: 'local' or 'gcp'")
while True:
    run_mode = input("Mode: 'local' or 'gcp'").strip().lower()
    if run_mode in {"local", "gcp"}:
        break
    print("Invalid mode. Please enter 'local' or 'gcp'.")


# ==== LOCAL MODE ====
if run_mode == "local":
    folder_path = Path("../Data/RAW_test/")
    pdf_files = sorted(folder_path.glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in: {folder_path.resolve()}")

    total_files = len(pdf_files)
    print(f"Found {total_files} PDF file(s) in {folder_path.resolve()}")

    while True:
        user_input = input(
            f"How many files do you want to process? (1-{total_files}, or 'all'): "
        ).strip().lower()

        if user_input == "all":
            num_to_process = total_files
            break

        if user_input in {"q", "quit", "exit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")

        try:
            num_to_process = int(user_input)
        except ValueError:
            print("Invalid input. Enter a positive integer, 'all', or 'q' to quit.")
            continue

        if 1 <= num_to_process <= total_files:
            break

        print(f"Please enter a value between 1 and {total_files}, or 'all'.")

    selected_pdf_files = pdf_files[:num_to_process]
    print(f"Selected {len(selected_pdf_files)} file(s) for processing.")

    dates_exstracted = run_extraction(selected_pdf_files)
    print(f"Processed {len(dates_exstracted)} PDF file(s) from: {folder_path.resolve()}")


# ==== GCP MODE ====
else:
    bucket_name = GCP_DEFAULT_BUCKET
    gcs_prefix = GCP_DEFAULT_PREFIX
    print(f"Using default GCP source from gcp_metrics_from_pdfs.py: gs://{bucket_name}/{gcs_prefix}")

    batch_size_raw = input("Batch size [10]: ").strip()
    batch_size = int(batch_size_raw) if batch_size_raw else 10

    max_batches_raw = input("Max batches (blank = all): ").strip()
    max_batches = int(max_batches_raw) if max_batches_raw else None

    keep_pdfs_raw = input("Keep downloaded PDFs locally? [y/N]: ").strip().lower()
    keep_pdfs = keep_pdfs_raw in {"y", "yes"}

    processor = GCPBatchProcessor(
        bucket_name=bucket_name,
        gcs_prefix=gcs_prefix,
        local_pdf_folder="thesis_pdfs",
        batch_size=batch_size,
    )

    dates_exstracted = processor.process_all_batches(
        keep_pdfs=keep_pdfs,
        max_batches=max_batches,
    )

    print(f"Processed {len(dates_exstracted)} PDF file(s) from gs://{bucket_name}/{gcs_prefix}")
    
# ==== SHOW RESULTS ====
# summary of results
num_found = sum(1 for entry in dates_exstracted if entry["extracted_month"])
num_total = len(dates_exstracted)
print(f"\nExtracted hand-in month from {num_found}/{num_total} PDFs ({(num_found / num_total * 100) if num_total > 0 else 0:.1f}%)\n")

Choose extraction mode: 'local' or 'gcp'
Using default GCP source from gcp_metrics_from_pdfs.py: gs://thesis_archive_bucket/dtu_findit/master_thesis/


2026-03-21 12:35:39,606 - seasonality_extractor - INFO - Initialising GCP Storage client...
/Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD

Processed 6332 PDF file(s) from gs://thesis_archive_bucket/dtu_findit/master_thesis/

Extracted hand-in month from 5038/6332 PDFs (79.6%)



## EXPORT CSV

In [7]:
# ==== EXPORT RESULTS ====
save_outputs(dates_exstracted)
print(f"CSV written to: {OUTPUT_CSV}")

CSV written to: /Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/exstraction_more_from_pdf/handin_month_summary.csv


## JOIN SEASONALITY WITH METRICS

In [ ]:
seasonality_df = pd.read_csv("../Data/gcp_order/dtu_findit/extraction_and_processing/handin_month_summary.csv", dtype=str)

ParserError: Error tokenizing data. C error: Expected 2 fields in line 13, saw 5


In [ ]:
import pandas as pd
from pandas.errors import ParserError

metrics_path = Path("../Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis.csv")

# robust CSV read
try:
    metrics_df = pd.read_csv(metrics_path, dtype=str)
except ParserError:
    # fallback for malformed rows (e.g., unescaped commas)
    metrics_df = pd.read_csv(
        metrics_path,
        dtype=str,
        low_memory=False,
        engine="python",
        on_bad_lines="warn",  # skip bad lines but warn
    )

print(metrics_df.shape)
display(metrics_df.head())


ValueError: The 'low_memory' option is not supported with the 'python' engine

In [ ]:
#read csv
import pandas as pd
from pathlib import Path


metrics_df = pd.read_csv(metrics_path, sep=",", dtype=str, low_memory=False)

display(metrics_df.head())

ParserError: Error tokenizing data. C error: Expected 7 fields in line 3, saw 17


## PRINT ALL RESULTS

In [ ]:
# print all results in a readable format
for entry in dates_exstracted:
    print(f"{entry['filename']}:\n{entry['extracted_month']}\n")

# OLD PRODUCTION

In [ ]:
# ==== IMPORTS ====
from __future__ import annotations

import json
import re
from pathlib import Path

import dateparser
import pdfplumber


In [ ]:
# ==== CHOOSE FILES TO PROCESS ====
folder_path = Path("../Data/RAW_test/")
pdf_files = sorted(folder_path.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in: {folder_path.resolve()}")

total_files = len(pdf_files)
print(f"Found {total_files} PDF file(s) in {folder_path.resolve()}")

while True:
    user_input = input(
        f"How many files do you want to process? (1-{total_files}, or 'all'): "
    ).strip().lower()

    if user_input == "all":
        num_to_process = total_files
        break

    if user_input in {"q", "quit", "exit"}:
        raise SystemExit("Execution terminated by user. No files were processed.")

    try:
        num_to_process = int(user_input)
    except ValueError:
        print("Invalid input. Enter a positive integer, 'all', or 'q' to quit.")
        continue

    if 1 <= num_to_process <= total_files:
        break

    print(f"Please enter a value between 1 and {total_files}, or 'all'.")

selected_pdf_files = pdf_files[:num_to_process]
print(f"Selected {len(selected_pdf_files)} file(s) for processing.")